# Phase 9 — Production Recommendation Serving & API Layer

## 1. Title & Objective
This notebook demonstrates the **Production Recommendation Serving & API Layer** (`src/api.py`), providing a RESTful API wrapper around the completed Phase 1–8 recommendation system using **FastAPI**, **Pydantic**, and **TestClient**.

**Note**: This notebook demonstrates API serving; it is not a cloud deployment or production infrastructure demonstration.

## 2. Why an API Layer?
Machine learning models deliver business value when integrated into application workflows. A REST API provides:
- **Decoupling**: Separates client/application interface logic from core machine learning inference.
- **Programmatic Access**: Enables standard HTTP requests and structured JSON response payloads.
- **Input Validation**: Ensures type-safe, schema-validated request parsing via Pydantic.
- **Integration Readiness**: Allows future web frontends, mobile apps, or microservices to interface with the recommendation system without depending on internal Python objects.

## 3. Existing Recommendation Architecture
Phase 9 serves the complete pipeline without modifying underlying recommendation algorithms:

```
Client Request  ──►  FastAPI  ──►  Pydantic Validation  ──►  ModelContainer  ──►  Phase 1–8 Recommendation Engines  ──►  JSON Sanitization  ──►  Client Response
```

- **Phase 3 (Content)**: Single-movie content recommendations via TF-IDF vectorization and Cosine Similarity.
- **Phase 4 (Personalization)**: User profile content recommendations using weighted rating history profile vector $\mathbf{u}$.
- **Phase 6 (Collaborative Filtering)**: Item-based collaborative filtering using MovieLens latest-small interaction matrix.
- **Phase 7 (Hybrid)**: Candidate pool union ($N_{\text{cand}}=100$), Min-Max score normalization, and weighted linear fusion.
- **Phase 8 (Explainability)**: Transparent recommendation evidence breakdown (content overlaps, collaborative similarity, hybrid weights, summary).
- **Phase 9 (API Serving)**: Production-style REST API endpoints.

## 4. FastAPI Application Setup
We import the FastAPI application instance `app` from `src.api` and initialize `fastapi.testclient.TestClient` for in-process request execution.

In [ ]:
import os
import sys
from pathlib import Path
from fastapi.testclient import TestClient

# Robust project root resolution for notebook environment or CLI execution
current_dir = Path(os.getcwd())
project_root = current_dir if (current_dir / "data").exists() else current_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.api import app, init_models

print("Initializing API models and FastAPI TestClient...")
models = init_models()
client = TestClient(app)
print("FastAPI app & TestClient initialized successfully.")

## 5. Health Check
We invoke `GET /health` to inspect service readiness and loaded dataset metadata.

In [ ]:
import json
response = client.get("/health")
print(f"GET /health Status Code: {response.status_code}")
print("Response Payload:")
print(json.dumps(response.json(), indent=2))

## 6. Content Recommendation API
We query `GET /recommend/content?title=Avatar&top_n=5` to retrieve single-movie content recommendations.

In [ ]:
response = client.get("/recommend/content?title=Avatar&top_n=5")
print(f"GET /recommend/content Status Code: {response.status_code}")
print("Response Payload:")
print(json.dumps(response.json(), indent=2))

## 7. Personalized Recommendation API
We send `POST /recommend/personalized` with a user rating history. The endpoint delegates recommendation generation to the Phase 4 `PersonalizedRecommender`.

In [ ]:
payload = {
    "history": [
        {"title": "Avatar", "rating": 5},
        {"title": "Aliens", "rating": 4}
    ],
    "top_n": 5
}
response = client.post("/recommend/personalized", json=payload)
print(f"POST /recommend/personalized Status Code: {response.status_code}")
print("Response Payload:")
print(json.dumps(response.json(), indent=2))

## 8. Hybrid Recommendation API
We query `POST /recommend/hybrid` with `user_id = 1`, `alpha = 0.5`, and `top_n = 5`. The endpoint delegates candidate union and weighted scoring to Phase 7 `HybridMovieRecommender`.

In [ ]:
payload = {
    "user_id": 1,
    "alpha": 0.5,
    "top_n": 5
}
response = client.post("/recommend/hybrid", json=payload)
print(f"POST /recommend/hybrid Status Code: {response.status_code}")
print("Response Payload:")
print(json.dumps(response.json(), indent=2))

## 9. Explainable Recommendation API
We query `POST /recommend/explain` to extract recommendation evidence from Phase 8 `RecommendationExplainer`.

**Note**: No LLM-generated explanation is used by this API.

In [ ]:
payload = {
    "user_id": 1,
    "target_movie_id": 2918,
    "alpha": 0.5
}
response = client.post("/recommend/explain", json=payload)
print(f"POST /recommend/explain Status Code: {response.status_code}")
print("Response Payload:")
print(json.dumps(response.json(), indent=2))

## 10. Request Validation & Error Handling
We demonstrate robust Pydantic request validation and structured error responses (HTTP 400 Bad Request, HTTP 404 Not Found, HTTP 422 Unprocessable Entity) with no raw stack traces exposed.

In [ ]:
print("--- Test 1: Unknown Movie Title (Expect HTTP 404) ---")
res = client.get("/recommend/content?title=UnknownMovieXYZ999&top_n=5")
print(f"Status: {res.status_code} | Payload: {res.json()}")

print("\n--- Test 2: Invalid top_n <= 0 (Expect HTTP 422) ---")
res = client.get("/recommend/content?title=Avatar&top_n=0")
print(f"Status: {res.status_code} | Payload: {res.json()}")

print("\n--- Test 3: Invalid Alpha > 1.0 (Expect HTTP 422) ---")
res = client.post("/recommend/hybrid", json={"user_id": 1, "alpha": 1.5, "top_n": 5})
print(f"Status: {res.status_code} | Payload: {res.json()}")

## 11. API Limitations & Production Considerations
This implementation is a **production-style serving demonstration**, not a deployed production service. Specific operational limitations include:
1. **In-process serving**: Executed in-memory via FastAPI TestClient / Uvicorn.
2. **Single-node in-memory model state**: Models and feature matrices reside in single-node memory (`ModelContainer` singleton).
3. **No authentication/JWT**: API endpoints are unauthenticated.
4. **No persistent user database**: Rating histories are transient.
5. **No Redis caching**: Candidates are calculated on demand.
6. **No Docker/container deployment**: Kept lightweight within python workspace.
7. **No cloud deployment**: Local execution environment.
8. **No production monitoring**: Basic log/status endpoints.
9. **No load balancing**: Single ASGI worker process.
10. **No guaranteed production latency**: Bound by local CPU computation.

**Dataset & Model Limitations**:
- Title identity alignment between MovieLens latest-small and TMDB 5000 is fixed at 2,812 mapped movies (28.86% coverage).
- True new-user cold start remains unresolved because personalized content profiles require user rating history.
- As demonstrated in Phase 7 benchmark evaluation, pure item-based collaborative filtering ($NDCG@10 = 0.0954$) outperformed tested hybrid configurations on the temporal dataset.

## 12. Conclusion
Phase 9 successfully adds an API serving layer without changing the underlying recommendation algorithms.